In [21]:
import json
import pandas as pd

def load_json(file):
    with open(file) as fp:
        data = json.load(fp)
    return data

def write_json(dictionary, file):
    json_object = json.dumps(dictionary, indent=4)
 
    with open(file, "w") as outfile:
        outfile.write(json_object)

cache = load_json('hse-cache.json')

In [61]:
edited_or_new = load_json('national-sact-regimens_edited_or_new_rows.json')
edited_or_removed = load_json('national-sact-regimens_edited_or_removed_rows.json')

edited_or_new[0]

{'Regimen label': 'Breast SACT Regimens',
 'Regimen Name': 'Ribociclib Therapy (Adjuvant) - 28 dayRegimen',
 'Indication': '00892aIn combination with an aromatase inhibitor (AI) is indicated for the adjuvant treatment of patients with hormone receptor positive (HR+), human epidermal growth factor receptor 2 (HER2)-negative early breast cancer at high risk of recurrence.'}

In [62]:
edited_or_new[0]

{'Regimen label': 'Breast SACT Regimens',
 'Regimen Name': 'Ribociclib Therapy (Adjuvant) - 28 dayRegimen',
 'Indication': '00892aIn combination with an aromatase inhibitor (AI) is indicated for the adjuvant treatment of patients with hormone receptor positive (HR+), human epidermal growth factor receptor 2 (HER2)-negative early breast cancer at high risk of recurrence.'}

In [63]:
edited_or_removed[1]

{'Regimen label': 'Breast SACT Regimens',
 'Regimen Name': 'Ribociclib Therapy-28 daysRegimen',
 'Indication': '00525aTreatment of postmenopausal women with hormone receptor (HR) positive, human epidermal growth factor receptor 2 (HER2) negative locally advanced or metastatic breast cancer as initial endocrine-based therapy in combination with an aromatase inhibitor.00525bTreatment of women with hormone receptor (HR) positive, human epidermal growth factor receptor 2 (HER2) negative locally advanced or metastatic breast cancer in combination with an aromatase inhibitor or fulvestrant as initial endocrine based therapy or in women who have received prior endocrine therapy.In pre or perimenopausal women, the endocrine therapy should be combined with a luteinising hormone releasing hormone (LHRH) agonist.'}

In [64]:
len(edited_or_new)

172

# Create reconciliation table
Create a reconciliation table that includes the following columns:
- pre_Drug
- pre_Approved Indications
- pre_max_cosine
- post_Drug
- post_Approved Indications
- post_max_cosine
- drug match
- approved indications match
- could include id based on list index
- add a flag if more than one post was above the specified threshold

In [69]:
from typing import Dict, Any, List, Tuple
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors


def normalize_text(s: str) -> str:
    """
    Lowercase, remove ™/®, punctuation, and extra spaces.
    Keep letters/numbers so 'Encorafenib + binimetinib' ~ 'encorafenib and binimetinib'.
    """
    if not isinstance(s, str):
        s = str(s)
    s = s.lower().replace("®", " ").replace("™", " ")
    s = re.sub(r"[+/&]", " ", s)          # unify separators like + / &
    s = re.sub(r"[^a-z0-9\s]", " ", s)    # drop punctuation
    s = re.sub(r"\s+", " ", s).strip()
    return s


def build_nn_index(
    comp_list: List[str],
    analyzer: str = "char_wb",
    ngram_range: Tuple[int, int] = (2, 5),
):
    """
    Fit a TF-IDF vectorizer on normalized dictionary strings and train a cosine k-NN index.
    Returns a tiny bundle you pass to `find_neighbors`.
    """
    ids = range(0, len(comp_list))
    texts = [item[2] for item in comp_list]
    texts_norm = [normalize_text(t) for t in texts]

    vec = TfidfVectorizer(analyzer=analyzer, ngram_range=ngram_range, min_df=1, dtype=float)
    X = vec.fit_transform(texts_norm)              # matrix: len(dict) x vocab_size

    nn = NearestNeighbors(metric="cosine")
    nn.fit(X)                                      # index on TF-IDF vectors

    return {
        "vectorizer": vec, 
        "matrix": X, 
        "nn": nn, 
        "ids": ids, 
        "texts": texts, 
        "texts_norm": texts_norm,
        "original_Drug": [item[0] for item in comp_list],
        "original_Indications": [item[1] for item in comp_list]
    }


def find_neighbors(index_bundle, query: str, k: int = 5) -> List[Tuple[Any, str, float]]:
    """
    Return a list of (id, original_text, similarity) sorted by similarity (desc).
    Cosine similarity = 1 - cosine distance from NearestNeighbors.
    """
    #q_norm = query
    q_norm = normalize_text(query)
    q_vec = index_bundle["vectorizer"].transform([q_norm])
    distances, indices = index_bundle["nn"].kneighbors(q_vec, n_neighbors=min(k, len(index_bundle["texts"])))

    out = []
    for dist, idx in zip(distances[0], indices[0]):
        sim = 1.0 - float(dist)
        rtxt = index_bundle["texts"][idx]
        odrug = index_bundle["original_Drug"][idx]
        oindications = index_bundle["original_Indications"][idx]
        out.append((odrug, oindications, sim))

    out.sort(key=lambda x: x[2], reverse=True)
    return out

def create_obj_string(row, drug_label, indication_label):
    return f"Drug: {row[drug_label]}; Approved Indications: {row[indication_label]}"

In [78]:
#mode = "reimbursement"
mode = "national-sact-regimens"
name = f"{mode}.edited_or_new"
objs_to_index = edited_or_removed
objs_to_compare = edited_or_new
save_cache_reimbursement = False
save_cache_nsr = True

if mode == "reimbursement":
    drug_label = "Drug"
    indication_label = "Approved Indications"
elif mode == "national-sact-regimens":
    drug_label = "Regimen Name"
    indication_label = "Indication"
else:
    print("labels not specified")

vectors = []
for item in objs_to_index:
    value = create_obj_string(row=item, drug_label=drug_label, indication_label=indication_label)
    vectors.append((item[drug_label], item[indication_label], value))
idx = build_nn_index(comp_list=vectors)

threshold = 0.8
edited = []
removed = []
uncertain = []
results = []

for entry in objs_to_compare:
    value = create_obj_string(row=entry, drug_label=drug_label, indication_label=indication_label)
    result = find_neighbors(index_bundle=idx, query=value, k=5)
    close_matches = [r for r in result if r[2] >= threshold]

    record = {
        'Drug (pre)': entry[drug_label],
        'Approved Indications (pre)': entry[indication_label],
        'Drug (post)': None,
        'Approved Indications (post)': None,
        'max cosine': None,
        'n above cosine threshold': None,
        'match - drug': None,
        'match - approved indications': None,
        'results': {}
    }
    
    if len(close_matches) > 1:
        uncertain.append((entry, result))

        record['max cosine'] = max([item[2] for item in result])
        record['n above cosine threshold'] = len([item[2] for item in result if item[2] >= threshold])
        
    elif len(close_matches) == 1:
        match = close_matches[0]
        record['Drug (post)'] = match[0]
        record['Approved Indications (post)'] = match[1]
        record['max cosine'] = match[2]
        record['n above cosine threshold'] = len([item[2] for item in result if item[2] >= threshold])
        record['match - drug'] = entry[drug_label] == match[0]
        record['match - approved indications'] = entry[indication_label] == match[1]

    elif len(close_matches) < 1:
        uncertain.append((prior_entry, result))

        record['max cosine'] = max([item[2] for item in result])
        record['n above cosine threshold'] = len([item[2] for item in result if item[2] >= threshold])
    
    record['results'] = result

    if save_cache_reimbursement:
        record['Drug'] = entry[drug_label]
        record['Effective date for reimbursement'] = ""
        record['Funding stream'] = "ODMS"
        record['Approved Indications'] = entry[indication_label]
        record['biomarker'] = None
        record['cataloged'] = None

    if save_cache_nsr:
        record['preallocated'] = {
            'Regimen label': entry['Regimen label'],
            'Regimen Name': entry[drug_label],
            'Indication': entry[indication_label],
            'biomarker': None,
            'cataloged': None,
            'comment': ""
        }
        
    results.append(record)
results = pd.DataFrame(results)

results.to_csv(f"{name}.summary.txt", sep='\t', index=False)
write_json(dictionary=results.to_dict(orient='records'), file=f"{name}.summary.json")

/opt/miniconda3/envs/euro-moalmanac-db/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:2044: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. <class 'float'> 'dtype' will be converted to np.float64.
  warnings.warn(


In [73]:
print(len(edited_or_removed))

89


In [74]:
print(len(edited_or_new))

172


- pre_Drug
- pre_Approved Indications
- post_Drug
- post_Approved Indications
- max_cosine
- drug match
- approved indications match
- could include id based on list index
- add a flag if more than one post was above the specified threshold

# Preallocate new entries
Write a JSON file with the new entries